In [1]:
# DAY 1
import requests

# Fetch the main Remote OK page
url = "https://remoteok.com/remote-jobs"
response = requests.get(url)

# Check if the request was successful
if response.status_code == 200:
    print("Success! Downloaded the page.")
else:
    print(f"Error: {response.status_code}")
    
# Print first 500 characters of the HTML
print(response.text[:500])

# DAY 2
import requests
from bs4 import BeautifulSoup

url = "https://remoteok.com/api"
headers = {"User-Agent": "Mozilla/5.0"}

data = requests.get(url, headers=headers).json()[1:]

# Create simple HTML like the website structure
html = ""

for i, job in enumerate(data):
    html += f"""
    <tr data-offset="{i}">
        <td class="company position company_and_position">
            <a itemprop="url">
                <h2>{job['position']}</h2>
            </a>
        </td>
    </tr>
    """

soup = BeautifulSoup(html, "html.parser")

rows = soup.find_all("tr", attrs={"data-offset": True})
joblist=[]
for row in rows:
    titles = row.find_all("td", class_="company_and_position")
    for t in titles:
        link = t.find("a", attrs={"itemprop": "url"})
        if link:
            jobname = link.find("h2").text
            joblist.append(jobname)

print(joblist)

# Print first 10
for i, title in enumerate(joblist[:10]):
    print(f"{i+1}. {title}")

# DAY 3
import requests
from bs4 import BeautifulSoup
import json

url = "https://remoteok.com/api"
headers = {"User-Agent": "Mozilla/5.0"}

data = requests.get(url, headers=headers).json()[1:]

html = ""

# ---------- Create HTML Structure ----------
for i, job in enumerate(data):

    company = job.get("company", "Not Specified")
    location = job.get("location", "Not Specified")
    tags = job.get("tags", [])
    link = job.get("url", "")

    skills_html = ""

    for skill in tags:
        skills_html += f"""
        <a class='no-border tooltip-set action-add-tag'>
            <div>
                <h3>{skill}</h3>
            </div>
        </a>
        """

    html += f"""
    <tr data-offset="{i}" data-company="{company}" href="{link}">
        <td class="company position company_and_position">
            <div class="location tooltip">{location}</div>
        </td>

        <td class="tags">
            {skills_html}
        </td>
    </tr>
    """

# ---------- Parse HTML ----------
soup = BeautifulSoup(html, "html.parser")
rows = soup.find_all("tr", attrs={"data-offset": True})

joblist = []

# ---------- Extract Data ----------
for row in rows:

    # Company
    company = row.get("data-company", "Not Specified")

    # URL
    job_url = row.get("href", "Not Available")

    # Location
    loc_tag = row.find("div", class_="location")
    if loc_tag:
        location_text = loc_tag.text.strip()
        if "🌏" in location_text or "worldwide" in location_text.lower():
            location = "Worldwide"
        else:
            location = location_text
    else:
        location = "Not Specified"

    
    # Skills
    skills = []
    tag_section = row.find("td", class_="tags")

    if tag_section:
        skill_links = tag_section.find_all("a", class_="no-border tooltip-set action-add-tag")

        for link in skill_links:
            div = link.find("div")
            if div:
                h3 = div.find("h3")
                if h3:
                    skills.append(h3.text.strip())

    # Employment Type from JSON-LD
    emp_type = "Not Specified"
    script_tag = row.find("td", class_="image has-logo")
    if script_tag:
        json_ld = script_tag.find("script", type="application/ld+json")
        if json_ld:
            try:
                data = json.loads(json_ld.string)
                emp_type = data.get("employmentType", "Not Specified")
            except json.JSONDecodeError:
                emp_type = "Not Specified"


    # Combine
    job_data = {
        "company": company,
        "location": location,
        "employment_type": emp_type,
        "skills": skills,
        "url": job_url
    }

    joblist.append(job_data)

# ---------- Print Output ----------
print (joblist)

# DAY 4
import requests
from bs4 import BeautifulSoup
import time
import json
headers = {"User-Agent": "Mozilla/5.0"}

all_jobs = []

base_url = "https://remoteok.com/remote-jobs"

for page in range(1, 6):

    print(f"Scraping page {page}...")

    # Page URL logic
    if page == 1:
        url = base_url
    else:
        url = f"{base_url}/{page}"

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        rows = soup.find_all("tr", attrs={"data-offset": True})

        for row in rows:
            company = row.get("data-company", "Not Specified")
            job_url = row.get("data-href", "Not Available")
            loc_tag = row.find("div", class_="location")
            if loc_tag:
                location_text = loc_tag.text.strip()
                if "🌏" in location_text or "worldwide" in location_text.lower():
                    location = "Worldwide"
                else:
                    location = location_text
            else:
                location = "Not Specified"


            skills = []
            tag_section = row.find("td", class_="tags")
            if tag_section:
                skill_links = tag_section.find_all("a", class_="no-border tooltip-set action-add-tag")
                for link in skill_links:
                    div = link.find("div")
                    if div:
                        h3 = div.find("h3")
                        if h3:
                            skills.append(h3.text.strip())

            # Employment Type from JSON-LD
            emp_type = "Not Specified"
            script_tag = row.find("td", class_="image has-logo")
            if script_tag:
                json_ld = script_tag.find("script", type="application/ld+json")
                if json_ld:
                    try:
                        data = json.loads(json_ld.string)
                        emp_type = data.get("employmentType", "Not Specified")
                    except json.JSONDecodeError:
                        emp_type = "Not Specified"

            job_data = {
        "company": company,
        "location": location,
        "skills": skills,
        "employment_type": emp_type,
        "url": job_url
    }

            all_jobs.append(job_data)

        # Respect crawl delay
        time.sleep(1)

    except requests.exceptions.RequestException as e:
        print(f"Error on page {page}: {e}")
        continue

print(f"\nTotal jobs scraped: {len(all_jobs)}")

# DAY 5
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

headers = {"User-Agent": "Mozilla/5.0"}

all_jobs = []

base_url = "https://remoteok.com/remote-jobs"

for page in range(1, 6):

    print(f"Scraping page {page}...")

    # Page URL logic
    if page == 1:
        url = base_url
    else:
        url = f"{base_url}/{page}"

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        rows = soup.find_all("tr", attrs={"data-offset": True})

        for row in rows:
            company = row.get("data-company", "Not Specified")
            job_url = row.get("data-href", "Not Available")
            loc_tag = row.find("div", class_="location")
            if loc_tag:
                location_text = loc_tag.text.strip()
                if "🌏" in location_text or "worldwide" in location_text.lower():
                    location = "Worldwide"
                else:
                    location = location_text
            else:
                location = "Not Specified"


            skills = []
            tag_section = row.find("td", class_="tags")
            if tag_section:
                skill_links = tag_section.find_all("a", class_="no-border tooltip-set action-add-tag")
                for link in skill_links:
                    div = link.find("div")
                    if div:
                        h3 = div.find("h3")
                        if h3:
                            skills.append(h3.text.strip())

            job_data = {
        "company": company,
        "location": location,
        "skills": skills,
        "url": job_url
    }

            all_jobs.append(job_data)

        # Respect crawl delay
        time.sleep(1)

    except requests.exceptions.RequestException as e:
        print(f"Error on page {page}: {e}")
        continue

# ---------- Convert list of dictionaries to DataFrame ----------
df = pd.DataFrame(all_jobs)

# ---------- Display basic info ----------
print(f"Total rows scraped: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nMissing values per column:\n{df.isnull().sum()}")

# ---------- Handle missing values ----------
# Replace empty strings or None with 'N/A'
df.fillna('N/A', inplace=True)

# Remove rows where critical fields are missing
df = df[(df['company'] != 'N/A') & (df['location'] != 'N/A') & (df['url'] != 'Not Available')]

# ---------- Remove duplicates ----------
# Based on company + location + url
df = df.drop_duplicates(subset=['company', 'location', 'url'])

# ---------- Clean text ----------
# Lowercase and strip whitespace
df['company'] = df['company'].str.lower().str.strip()
df['location'] = df['location'].str.lower().str.strip()

# For skills, join list into comma-separated string and lowercase
df['skills'] = df['skills'].apply(lambda x: ', '.join([s.lower().strip() for s in x]) if isinstance(x, list) else '')

# ---------- Save to CSV ----------
csv_filename = 'remoteok_jobs_cleaned.csv'
df.to_csv(csv_filename, index=False)

# ---------- Summary ----------
print(f"\nCleaned data saved to '{csv_filename}'. Final row count: {len(df)}")
print("\nSample data:")
print(df.head())

# DAY 6
# remoteok_full_pipeline.py
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns

# ---------- SETTINGS ----------
BASE_URL = "https://remoteok.com/remote-jobs"
HEADERS = {"User-Agent": "Mozilla/5.0"}
PAGES_TO_SCRAPE = 5  # adjust as needed
CSV_FILE = "remoteok_jobs_full.csv"

all_jobs = []

# ---------- SCRAPING ----------
for page in range(1, PAGES_TO_SCRAPE + 1):
    print(f"Scraping page {page}...")
    url = f"{BASE_URL}?page={page}"
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        rows = soup.find_all("tr", attrs={"data-offset": True})
        for row in rows:
            job = {}

            # Title
            td_tag = row.find("td", class_="company position company_and_position")
            h2_tag = td_tag.find("h2") if td_tag else None
            job_title = h2_tag.text.strip() if h2_tag else "N/A"

            # Company
            company = row.get("data-company", "N/A")

            # Location
            loc_div = td_tag.find("div", class_="location") if td_tag else None
            location = loc_div.text.strip() if loc_div else "Not Specified"

            # Skills
            skills_td = row.find("td", class_="tags")
            skills_list = []
            if skills_td:
                a_tags = skills_td.find_all("a", class_="no-border tooltip-set action-add-tag")
                for a in a_tags:
                    h3 = a.find("h3")
                    if h3:
                        skills_list.append(h3.text.strip())
            skills = ", ".join(skills_list) if skills_list else "N/A"

            # Employment type from JSON-LD
            emp_type = "Not Specified"
            img_td = row.find("td", class_="image has-logo")
            if img_td:
                script_tag = img_td.find("script", type="application/ld+json")
                if script_tag:
                    try:
                        data_json = json.loads(script_tag.string)
                        emp_type = data_json.get("employmentType", "Not Specified")
                    except:
                        pass

            # Job URL
            job_url = td_tag.find("a", itemprop="url").get("href") if td_tag and td_tag.find("a", itemprop="url") else "N/A"
            if job_url != "N/A" and not job_url.startswith("http"):
                job_url = "https://remoteok.com" + job_url

            # Add to dict
            job["title"] = job_title
            job["company"] = company
            job["location"] = location
            job["skills"] = skills
            job["employment_type"] = emp_type
            job["url"] = job_url

            all_jobs.append(job)

        time.sleep(1)

    except requests.exceptions.RequestException as e:
        print(f"Error on page {page}: {e}")
        continue

print(f"Scraped {len(all_jobs)} jobs.")

# ---------- SAVE CSV ----------
df = pd.DataFrame(all_jobs)
# Remove duplicates
df = df.drop_duplicates(subset=["title", "company", "url"])
# Clean text
df['title'] = df['title'].str.strip().str.lower()
df['company'] = df['company'].str.strip().str.lower()
df['location'] = df['location'].str.strip()
df['skills'] = df['skills'].str.strip()
df['employment_type'] = df['employment_type'].str.strip()
df.to_csv(CSV_FILE, index=False)
print(f"Saved CSV: {CSV_FILE}")

import pandas as pd

# Load CSV
df = pd.read_csv("remoteok_jobs_full.csv")

print(f"Total jobs loaded: {len(df)}\n")

# Top 10 skills
df_skills = df.copy()
df_skills['skills'] = df_skills['skills'].str.split(', ')
df_skills = df_skills.explode('skills')
df_skills = df_skills[df_skills['skills'].str.strip() != '']
print("Top 10 Skills:")
print(df_skills['skills'].value_counts().head(10))

# Job type distribution
print("\nJob Type Distribution:")
print(df['employment_type'].value_counts())

# Top 10 locations
print("\nTop 10 Locations:")
print(df['location'].value_counts().head(10))

# Top 10 companies
print("\nTop 10 Companies:")
print(df['company'].value_counts().head(10))

# Average number of skills per job
df['num_skills'] = df['skills'].apply(lambda x: len(str(x).split(', ')) if x else 0)
print(f"\nAverage skills per job: {df['num_skills'].mean():.2f}")


# DAY 7
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---------- Load CSV ----------
df = pd.read_csv("remoteok_jobs_full.csv")

# ---------- Setup ----------
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10,6)

# ---------- Top 10 Job Titles ----------
top_titles = df['title'].value_counts().head(10)
plt.figure()
sns.barplot(x=top_titles.values, y=top_titles.index, palette="viridis")
plt.title("Top 10 Job Titles")
plt.xlabel("Number of Jobs")
plt.ylabel("Job Title")
plt.tight_layout()
plt.show()

# ---------- Top 10 Companies ----------
top_companies = df['company'].value_counts().head(10)
plt.figure()
sns.barplot(x=top_companies.values, y=top_companies.index, palette="magma")
plt.title("Top 10 Companies by Job Postings")
plt.xlabel("Number of Jobs")
plt.ylabel("Company")
plt.tight_layout()
plt.show()

# ---------- Top 10 Locations ----------
top_locations = df['location'].value_counts().head(10)
plt.figure()
sns.barplot(x=top_locations.values, y=top_locations.index, palette="coolwarm")
plt.title("Top 10 Job Locations")
plt.xlabel("Number of Jobs")
plt.ylabel("Location")
plt.tight_layout()
plt.show()

# ---------- Top 10 Skills ----------
# Explode skills into individual rows
df_skills = df.copy()
df_skills['skills'] = df_skills['skills'].str.split(', ')
df_skills = df_skills.explode('skills')
df_skills = df_skills[df_skills['skills'].str.strip() != '']
top_skills = df_skills['skills'].value_counts().head(10)

plt.figure()
sns.barplot(x=top_skills.values, y=top_skills.index, palette="cubehelix")
plt.title("Top 10 Skills in Job Listings")
plt.xlabel("Number of Jobs")
plt.ylabel("Skill")
plt.tight_layout()
plt.show()

# ---------- Job Type Distribution ----------
job_types = df['employment_type'].value_counts()
plt.figure()
sns.barplot(x=job_types.index, y=job_types.values, palette="Set2")
plt.title("Job Type Distribution")
plt.xlabel("Employment Type")
plt.ylabel("Number of Jobs")
plt.tight_layout()
plt.show()

# ---------- Average Skills per Job ----------
df['num_skills'] = df['skills'].apply(lambda x: len(str(x).split(', ')) if x else 0)
plt.figure()
sns.histplot(df['num_skills'], bins=range(0,15), kde=False, color="skyblue")
plt.title("Distribution of Number of Skills per Job")
plt.xlabel("Number of Skills")
plt.ylabel("Number of Jobs")
plt.tight_layout()
plt.show()

Success! Downloaded the page.
<!doctype html><html lang="en" class="   pageType-frontpage  remoteok    minimize-header   catch-emails-enabled">	<head>
			


					<link rel="stylesheet" href="/global.css?1761481057">
					<script>
													var userIsAdmin=false;
											</script>
					<meta charset="UTF-8">
					<title>Remote Jobs in Programming, Design, Sales and more #OpenSalaries</title>
					<meta name="description" content="Looking for a remote job? Remote OK® is the #1 Remote Job Platform and has 1,130,568+ remot
['Early Career Software Engineer Warfighter Systems', 'Growth Engineer', 'Senior Software Engineer Prime', 'Technical Writer', 'Lead VFX Artist', 'Senior Facial Character TD', 'Social Media Video Creator Intern', 'AI Engineer I', 'Senior Security Engineer', 'Product Engineer', 'Senior 3D Artist Monopoly GO', 'Senior User Experience Designer', 'Product Designer Creators', 'Global Administrative Assistant', 'Chief of Staff', 'Senior Database Reliability Engineer', 'S

KeyError: 'title'